# Assignment 2 - Jonas Gstoettenmayr

In [2]:
import numpy as np

In [3]:
rng = np.random.default_rng(42)

In [4]:
sentences = [
    "Production on the film began in June 1921 with Colleen’s return from New York and Florida, where she was making The Lotus Eater with John Barrymore which was directed by Marshall Neilan.",
    "The device operates as long as the magic smoke is trapped inside of it, but when the smoke escapes from it, the device ceases to operate.",
    "The 1980s farm crisis caused some families to have to give up their farms, and farms have been merged to industrial scale.",
    "From conceptual structuration, the architecture on which the co-simulation framework is developed and the formal semantic relations/syntactic formulation are defined.",
    "The chest is a chestnut colour with thin black bars, while long black-margined plumes arise from its flanks."
]

## Exercise 1

**Exercise 1 Simple Data Augmentation**

Implement a simple data augmentation technique by randomly swapping words. Create an
algorithm that swaps a specified ratio of words in the sentence. Make sure to swap the same
word just once.

- Generate 5 augmented sentences from the input corpus using random swaps.

In [5]:
def swap_words(sentence: str, ratio: float = 0.1):
    words: list[str] = sentence.split()
    words_to_swap: int = int(len(words) * ratio) //2
    pool = list(np.arange(len(words)))
    for _ in range(words_to_swap):
        if len(pool) < 2:
            break
        a, b = rng.choice(pool, size=2, replace=False)#type:ignore
        pool.remove(a)
        pool.remove(b)
        words[a], words[b] = words[b], words[a]#type:ignore
    return " ".join(words)
        
for sentence in sentences:
    print(swap_words(sentence, 0.5))

Production on John Lotus where in Colleen’s 1921 with June return from which Neilan. and with began she was making The film Barrymore Florida, the Eater New was directed by Marshall York
The device inside as is as the magic smoke long trapped operates ceases it, but when from smoke escapes the it, operate. device of to the
give 1980s some crisis caused farm families and have industrial The up their been to farms have farms, merged to to scale.
From conceptual formal the architecture on the are and framework is developed co-simulation which structuration, formulation relations/syntactic semantic the defined.
The colour thin a chestnut chest bars, is black with while plumes black-margined long arise from its flanks.


## Exercise 2

**Exercise 2 Synonym Replacement Augmentation**

Use WordNet from NLTK to replace words
with synonyms. Make the ratio of word replacements configurable.

- Retrieve synonyms for selected words
- Replace words in a sentence with synonyms
- Generate at least 3 augmented versions of the input corpus

In [6]:
import nltk
nltk.download("wordnet")
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jojog\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
def replace_with_synonmys(sentence: str, ratio: float = 0.1) -> str:
    words: list[str] = sentence.split()
    words_to_swap: int = int(len(words) * ratio)
    swapped = 0
    pool = list(np.arange(len(words)))
    while swapped < words_to_swap and len(pool) != 0:
        a = rng.choice(pool)#type:ignore
        pool.remove(a)
        # synsets = wn.synsets(words[a].lower())
        syns = {l.name().replace("_", " ") for s in wn.synsets(words[a]) for l in s.lemmas() if l.name().lower() != words[a].lower()}#type:ignore
        if syns:
            swapped += 1
            words[a] = rng.choice(list(syns))#type:ignore
    return " ".join(words)

In [8]:
[[replace_with_synonmys(sentence) for sentence in sentences] for _ in range(3)]

[['Production on the film began in June 1921 with Colleen’s restoration from New York and Florida, where she was making The Lotus feeder with John Barrymore which comprise directed by Marshall Neilan.',
  'The device operates as long as the magic smoke is trapped inside of it, but when the smoke escapes from it, the gimmick end to operate.',
  'The 1980s produce crisis caused more or less families to have to give up their farms, and farms have been merged to industrial scale.',
  'From conceptual structuration, the architecture on which the co-simulation framework is educate and the evening gown semantic relations/syntactic formulation are defined.',
  'The chest is a chestnut tree colour with thin black bars, while long black-margined plumes arise from its flanks.'],
 ['Production on the cinema began in June 1921 with Colleen’s return from New York and Florida, where she Evergreen State making The Lotus Eater with toilet Barrymore which was directed by Marshall Neilan.',
  'The device

## Exercise 3

**Exercise 3 Back-Translation Augmentation**

Use the transformers package to augment data by back-translation. The following provides a function to use transformers to translate. There are many models on HuggingFace, on possible
option is Helsinki-NLP/opus-mt_tiny_eng-fra.

- Augment the input corpus by backtranslation.
- Generate at least 3 augmented versions by using different languages

In [9]:
from transformers import MarianMTModel, MarianTokenizer
model_name_to_eng = "Helsinki-NLP/opus-mt_tiny_fra-eng"
tokenizer_to_eng = MarianTokenizer.from_pretrained(model_name_to_eng)
model_to_eng = MarianMTModel.from_pretrained(model_name_to_eng)

model_name_to_fra = "Helsinki-NLP/opus-mt_tiny_eng-fra"
tokenizer_to_fra = MarianTokenizer.from_pretrained(model_name_to_fra)
model_to_fra = MarianMTModel.from_pretrained(model_name_to_fra)

Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [10]:
tok = tokenizer_to_fra("I am a baguette lover. Sips wine. Hon Hon Hon", return_tensors="pt").input_ids
output = model_to_fra.generate(tok)[0]
french = tokenizer_to_fra.decode(output, skip_special_tokens=True)
french

'Je suis un amant de baguette. Sips vin. Hon Hon Hon'

In [11]:
tok = tokenizer_to_eng(french, return_tensors="pt").input_ids
output = model_to_eng.generate(tok)[0]
tokenizer_to_eng.decode(output, skip_special_tokens=True)

'I am a baguette lover. Sips vin. Hon Hon Hon Hon Hon'

In [12]:
def roundtrip_translate(sentences, model_name_to, model_name_original):
    
    
    tokenizer_to = MarianTokenizer.from_pretrained(model_name_to)
    model_to = MarianMTModel.from_pretrained(model_name_to)
    
    tokenizer_original = MarianTokenizer.from_pretrained(model_name_original)
    model_original = MarianMTModel.from_pretrained(model_name_original)
    
    results = []
    for sentence in sentences:
        inputs_a = tokenizer_to(sentence, return_tensors="pt")
        outputs_a = model_to.generate(**inputs_a)
        translated = tokenizer_to.decode(outputs_a[0], skip_special_tokens=True)
        
        inputs_b = tokenizer_original(translated, return_tensors="pt")
        outputs_b = model_original.generate(**inputs_b)
        back_to_original = tokenizer_original.decode(outputs_b[0], skip_special_tokens=True)
        results.append(back_to_original)
    return results
        
roundtrip_translate(sentences, "Helsinki-NLP/opus-mt_tiny_eng-fra", "Helsinki-NLP/opus-mt_tiny_fra-eng")

Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


['Production on the film began in June 1921 with the return of Colleen from New York and Florida, where she was doing The Lotus Eater with John Barrymore, directed by Marshall Neilan.',
 'The device works as long as the magic smoke is trapped inside it, but when the smoke escapes from it, the device stops working.',
 'The agricultural crisis of the 1980s led some families to abandon their farms, and farms were merged on an industrial scale.',
 'From the conceptual structure, the architecture on which the co-simulation framework is developed and formal semantic relationships/creatctics are defined.',
 'The chest is a chestnut color with thin black bars, while long black plumes come from its flanks.']

In [13]:
to_languages = ["Helsinki-NLP/opus-mt_tiny_eng-fra", "Helsinki-NLP/opus-mt_tiny_eng-deu", "Helsinki-NLP/opus-mt_tiny_eng-rus"]
from_languages = ["Helsinki-NLP/opus-mt_tiny_fra-eng", "Helsinki-NLP/opus-mt_tiny_deu-eng", "Helsinki-NLP/opus-mt_tiny_rus-eng"]
languaes = zip(to_languages, from_languages)

In [14]:
result = [roundtrip_translate(sentences, to, fro) for (to, fro)in languaes]
result

Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/151 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[['Production on the film began in June 1921 with the return of Colleen from New York and Florida, where she was doing The Lotus Eater with John Barrymore, directed by Marshall Neilan.',
  'The device works as long as the magic smoke is trapped inside it, but when the smoke escapes from it, the device stops working.',
  'The agricultural crisis of the 1980s led some families to abandon their farms, and farms were merged on an industrial scale.',
  'From the conceptual structure, the architecture on which the co-simulation framework is developed and formal semantic relationships/creatctics are defined.',
  'The chest is a chestnut color with thin black bars, while long black plumes come from its flanks.'],
 ["The production on the film began in June 1921 with Colleen's return from New York and Florida, where she made The Lotus Eater with John Barrymore, led by Marshall Neilan.",
  'The device works as long as the magic smoke is trapped in it, but when the smoke escapes from it, the devi

## Exercise 4

**Exercise 4 Basic Tokenization**

 Write a Python script that performs word and sentence tokenization on the input corpus using NLTK.

- Install the required libraries (nltk).
- Download tokenizer resources using nltk.download().
- Tokenize the text

**Task**

- Perform sentence tokenization
- Perform word tokenization
- Print the resulting tokens of the input corpus

In [15]:
from nltk.tokenize import word_tokenize, sent_tokenize

In [16]:
for sentence in sentences:
    print(word_tokenize(sentence))

['Production', 'on', 'the', 'film', 'began', 'in', 'June', '1921', 'with', 'Colleen', '’', 's', 'return', 'from', 'New', 'York', 'and', 'Florida', ',', 'where', 'she', 'was', 'making', 'The', 'Lotus', 'Eater', 'with', 'John', 'Barrymore', 'which', 'was', 'directed', 'by', 'Marshall', 'Neilan', '.']
['The', 'device', 'operates', 'as', 'long', 'as', 'the', 'magic', 'smoke', 'is', 'trapped', 'inside', 'of', 'it', ',', 'but', 'when', 'the', 'smoke', 'escapes', 'from', 'it', ',', 'the', 'device', 'ceases', 'to', 'operate', '.']
['The', '1980s', 'farm', 'crisis', 'caused', 'some', 'families', 'to', 'have', 'to', 'give', 'up', 'their', 'farms', ',', 'and', 'farms', 'have', 'been', 'merged', 'to', 'industrial', 'scale', '.']
['From', 'conceptual', 'structuration', ',', 'the', 'architecture', 'on', 'which', 'the', 'co-simulation', 'framework', 'is', 'developed', 'and', 'the', 'formal', 'semantic', 'relations/syntactic', 'formulation', 'are', 'defined', '.']
['The', 'chest', 'is', 'a', 'chestnut

In [17]:
merge = "\n".join(sentences)
print(merge)

Production on the film began in June 1921 with Colleen’s return from New York and Florida, where she was making The Lotus Eater with John Barrymore which was directed by Marshall Neilan.
The device operates as long as the magic smoke is trapped inside of it, but when the smoke escapes from it, the device ceases to operate.
The 1980s farm crisis caused some families to have to give up their farms, and farms have been merged to industrial scale.
From conceptual structuration, the architecture on which the co-simulation framework is developed and the formal semantic relations/syntactic formulation are defined.
The chest is a chestnut colour with thin black bars, while long black-margined plumes arise from its flanks.


In [18]:
sent_tokenize(merge)

['Production on the film began in June 1921 with Colleen’s return from New York and Florida, where she was making The Lotus Eater with John Barrymore which was directed by Marshall Neilan.',
 'The device operates as long as the magic smoke is trapped inside of it, but when the smoke escapes from it, the device ceases to operate.',
 'The 1980s farm crisis caused some families to have to give up their farms, and farms have been merged to industrial scale.',
 'From conceptual structuration, the architecture on which the co-simulation framework is developed and the formal semantic relations/syntactic formulation are defined.',
 'The chest is a chestnut colour with thin black bars, while long black-margined plumes arise from its flanks.']

## Exercise 5

**Exercise 5 Tokenization with spaCy**


Use spaCy to tokenize text and analyze linguistic features.

- Print all tokens
- Print token POS tags
- Print token lemmas

In [19]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [20]:
text = "Text to tokenize."
doc = nlp(text)
doc

Text to tokenize.

In [21]:
for token in doc:
    print(token)
    print(token.pos_)
    print(token.lemma_)
    print()

Text
NOUN
text

to
PART
to

tokenize
VERB
tokenize

.
PUNCT
.



In [22]:
for sentence in sentences:
    doc = nlp(sentence)
    print(sentence)
    for token in doc:
        print(token, end=" ")
    print()
    for token in doc:
        print(token.pos_, end=" ")
    print()
    for token in doc:
        print(token.lemma_, end=" ")
    print("\n")
    

Production on the film began in June 1921 with Colleen’s return from New York and Florida, where she was making The Lotus Eater with John Barrymore which was directed by Marshall Neilan.
Production on the film began in June 1921 with Colleen ’s return from New York and Florida , where she was making The Lotus Eater with John Barrymore which was directed by Marshall Neilan . 
NOUN ADP DET NOUN VERB ADP PROPN NUM ADP PROPN PART NOUN ADP PROPN PROPN CCONJ PROPN PUNCT SCONJ PRON AUX VERB DET PROPN PROPN ADP PROPN PROPN PRON AUX VERB ADP PROPN PROPN PUNCT 
production on the film begin in June 1921 with Colleen ’s return from New York and Florida , where she be make the Lotus Eater with John Barrymore which be direct by Marshall Neilan . 

The device operates as long as the magic smoke is trapped inside of it, but when the smoke escapes from it, the device ceases to operate.
The device operates as long as the magic smoke is trapped inside of it , but when the smoke escapes from it , the devi

## Exercise 6

**Exercise 6 Comparing Tokenization Methods on IMDb (NLTK, spaCy, and BPE)**

Compare three tokenization strategies for sentiment classification on the IMDb dataset on
eLearning:

1. NLTK word tokenization
2. spaCy tokenization
3. The GPT 2 PBE tokenizer.

We can easily use the GTP2 PBE tokenizer with the AutoTokenizer in transformers package.

Each tokenizer will be used inside an identical scikit-learn pipeline with TfidfVectorizerand LogisticRegression.

In [119]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import time
import pandas as pd

In [90]:
from transformers import AutoTokenizer
gpt_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [112]:
def spacy_tokenizer(text):
    doc = nlp.make_doc(text)
    return [token.text for token in doc]

In [103]:
data = pd.read_feather(path="../data/EX2/sentiments.feather")
X, y = data.iloc[:, 0].to_numpy(), data.iloc[:, 1].to_numpy()# on 10k because otherwise it takes to long
X_train, X_test, y_train, y_test, = train_test_split(X, y, test_size=0.2)

In [114]:
def build_pipeline(tokenizer_fn):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            tokenizer=tokenizer_fn,
            lowercase=True,
            token_pattern=None  # disable default tokenization
        )),
        ("clf", LogisticRegression(
            max_iter=200,
            solver="lbfgs"
        ))
    ])


In [128]:
tokenizers = [gpt_tokenizer.tokenize, nltk.word_tokenize, spacy_tokenizer]
pipelines = [build_pipeline(tokenizer) for tokenizer in tokenizers]

In [129]:
names = ["GPT", "NLTK", "spaCy"]
for (pipe, name) in zip(pipelines, names):
    start = time.time()
    print(f"Training pipe {name}...", end=" ")
    pipe.fit(X_train, y_train)
    print(f"took: {time.time()-start:.2f}s")

Training pipe GPT... took: 27.77s
Training pipe NLTK... took: 48.70s
Training pipe spaCy... took: 35.16s


In [ ]:
for (pipe, name) in zip(pipelines, names):
    start = time.time()
    pred = pipe.predict(X_test)
    t = time.time()-start
    
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    
    vocab = pipe.named_steps['tfidf'].vocabulary_
    
    print(f"{name} achieved:")
    print(f"\t accuracy of {acc:.2f}\n\t f1 {f1:.2f} on test score")
    print(f"\t took {t:.2f}s total, {t/(len(y_test)/1000):.2f}s per 1000 samples")
    print(f"\t vocab siize: {len(vocab)}")

GPT achieved:
	 accuracy of 0.8984
	 f1 0.8980124472997391 on test score
	 took 8.00s total, 0.80s per 1000 samples
	 vocab siize: 29348
NLTK achieved:
	 accuracy of 0.8978
	 f1 0.8973483326637204 on test score
	 took 11.19s total, 1.12s per 1000 samples
	 vocab siize: 145054
spaCy achieved:
	 accuracy of 0.8986
	 f1 0.8983356727491478 on test score
	 took 8.80s total, 0.88s per 1000 samples
	 vocab siize: 133882


In [137]:
for (tokenizer, name) in zip(tokenizers, names):
    token_counts = [len(tokenizer(text)) for text in X]
    token_avg = np.mean(token_counts)#type:ignore
    print(f"{name} has token avg: {token_avg}")

GPT has token avg: 296.24864
NLTK has token avg: 279.4838
spaCy has token avg: 269.39262


When cutting out the large spacy pipeline and only using the tokenizer all 3 are pretty much euqal in (almost) all metrics.
the clear winner tough is GPT while the differences are not large it has the smallest vocabulary size, the fastest inference AND training time and the same accuracy as the other two.

## Exercise 7

**Exercise  Implementing a Byte Pair Encoding (BPE) Tokenizer**

Implement your own Byte Pair Encoding (BPE) tokenizer from scratch (no external BPE/tokenizer libraries). The tokenizer will consist of two main components:

- Training: learn merge rules from the IMDb training corpus
- Tokenization: apply merges greedily to produce subword tokens

**Guidelines for Your Implementation:**

1. Represent each unique word as a character sequence with an end-of-word marker, e.g.movie → m o v i e </w>.
2. Count symbol-pair frequencies across the full corpus vocabulary.
3. Iteratively merge the most frequent pair, updating the vocabulary after each merge.
4. Continue until you reach your target vocabulary size (e.g. 2000 merges).
5. Implement a tokenize() method that applies the learned merges greedily.

Tasks

- Load the IMDb dataset (train split only for BPE training).
- Implement the BPE training procedure.
- Implement the BPE tokenization procedure.
- Test the tokenizer on sample input sentences.

i made a very simple one that only deals with lowercase asci and ignores things like ."?...

In [ ]:
from collections import defaultdict

**AI usage: Gemini 3.1 FAST**
Promopt:
is my function for finding a Byte Pair encoding vocabulary correct?

```
def train_BPE(corpus: list[str], merges: int = 2000)-> set[str]:
    counter: defaultdict[str, int] = defaultdict(int)
    vocab = set(ascii_lowercase)
    
    corpus = [text.strip().lower() for text in corpus]
    
    for text in corpus:
        for char in text:
            if char in vocab:
                counter[char] += 1
    
    for _ in range(merges):
        srtd = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        # the original algorithm only takes one combination, but this makes it faster and easier, so for my scuffed version it is sufficient
        for i in range(len(srtd)-1):
            merge = set([srtd[i][0]+srtd[i+1][0], srtd[i+1][0]+srtd[0][0]])
            merge_len = len(srtd[i][0]+srtd[i+1][0])
            if len(merge.intersection(vocab)) == 0:
                break
            if i == len(srtd)-1:
                return vocab
    
        for text in corpus:
            for word in text.split():
                if len(word) < merge_len: continue
                if len(word) == merge_len:
                    if word in merge:
                        counter[word] += 1
                for i in range(0, len(word)-merge_len):
                    comb = text[i:i+merge_len]
                    if comb in merge:
                        counter[comb] += 1
        for new_voc in merge:
            counter[srtd[0][0]] -= counter[new_voc]
            counter[srtd[1][0]] -= counter[new_voc]
            if counter[new_voc] >0:
                vocab.add(new_voc)
    
    return vocab
```

In [247]:
def train_BPE(corpus: list[str], num_merges: int = 2000) -> set[str]:

    vocab_counts: defaultdict[tuple[str, ...], int] = defaultdict(int)
    for text in corpus:
        words = text.strip().lower().split()
        for word in words:
            tokens = tuple(list(word) + ["</w>"])
            vocab_counts[tokens] += 1

    for _ in range(num_merges):
        pairs = defaultdict(int)
        for word_tuple, freq in vocab_counts.items():
            for i in range(len(word_tuple) - 1):
                pairs[word_tuple[i], word_tuple[i+1]] += freq
        
        if not pairs:
            break
            
        # Find the most frequent pair
        best_pair = max(pairs, key=pairs.get)
        
        # 3. Perform the merge in the corpus
        new_vocab_counts:defaultdict[tuple[str, ...], int] = {}
        for word_tuple, freq in vocab_counts.items():
            new_word = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and (word_tuple[i], word_tuple[i+1]) == best_pair:
                    new_word.append(best_pair[0] + best_pair[1])
                    i += 2
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            new_vocab_counts[tuple(new_word)] = freq
        
        vocab_counts = new_vocab_counts
    
    final_vocab: set[str] = set()
    for keys in vocab_counts.keys():
        final_vocab.update(keys)
    
    return final_vocab

In [264]:
def tokenize_BPE(corpus: list[str], vocab: set[str]) -> list[list[str]]:
    all_tokens= []
    
    for text in corpus:
        words = [word+"</w>" for word in text.strip().lower().split()]
        tokens=[]
        for word in words:
            start = 0
            while start < len(word):
                match = False
                for end in range(len(word),start, -1):
                    subword = word[start:end]
                    if subword in vocab:
                        tokens.append(subword)
                        start += len(subword)
                        match = True
                        break
                if match == False:
                    tokens.append("</MISSING>")
                    start += 1
        all_tokens.append(tokens)
    
    return all_tokens

In [250]:
vocab = train_BPE(X[:100], 1000)
len(vocab)

1050

In [268]:
for sentence in tokenize_BPE(sentences, vocab):
    print(sentence)

['pro', 'duc', 'tion</w>', 'on</w>', 'the</w>', 'film</w>', 'be', 'ga', 'n</w>', 'in</w>', 'ju', 'ne', '</w>', '19', '2', '1', '</w>', 'with</w>', 'col', 'le', 'en', '</MISSING>', 's</w>', 're', 'turn', '</w>', 'from</w>', 'new</w>', 'y', 'or', 'k</w>', 'and</w>', 'f', 'lo', 'ri', 'da', ',</w>', 'where</w>', 'she</w>', 'was</w>', 'making</w>', 'the</w>', 'lo', 't', 'us</w>', 'e', 'ater</w>', 'with</w>', 'joh', 'n</w>', 'b', 'ar', 'r', 'y', 'more</w>', 'which</w>', 'was</w>', 'direct', 'ed</w>', 'by</w>', 'mar', 'sh', 'all</w>', 'ne', 'il', 'an', '.</w>']
['the</w>', 'de', 'vi', 'ce</w>', 'op', 'er', 'ates</w>', 'as</w>', 'lo', 'n', 'g</w>', 'as</w>', 'the</w>', 'mag', 'ic</w>', 's', 'mo', 'ke</w>', 'is</w>', 'tra', 'pp', 'ed</w>', 'in', 'si', 'de</w>', 'of</w>', 'it', ',</w>', 'but</w>', 'when</w>', 'the</w>', 's', 'mo', 'ke</w>', 'es', 'cap', 'es</w>', 'from</w>', 'it', ',</w>', 'the</w>', 'de', 'vi', 'ce</w>', 'c', 'e', 'as', 'es</w>', 'to</w>', 'op', 'er', 'at', 'e.</w>']
['the</w>'